# UR5e MuJoCo 시뮬레이션과 teleop

`ri_motion_v5_py312` 커널을 선택하세요. 마지막 셀에서 관절 슬라이더, Quest 3, SpaceMouse 중 하나로 OnRobot RG2가 장착된 UR5e를 조작합니다.

In [ ]:
from pathlib import Path
import subprocess
import sys

cwd = Path.cwd().resolve()
for repo_root in (cwd, *cwd.parents):
    if (repo_root / 'vai_robotics' / 'ur5e' / 'sim.py').is_file():
        break
else:
    raise FileNotFoundError('VAI_AIWORKER 저장소 안에서 노트북을 열어 주세요.')

simulator = [sys.executable, '-m', 'vai_robotics.ur5e.sim']

### 모델과 관절 동작 확인

뷰어를 열기 전에 UR5e 모델을 불러와 작은 관절 명령을 적용하고 결과를 확인합니다.

In [ ]:
check = subprocess.run([*simulator, '--check'], cwd=repo_root, capture_output=True, text=True)
print(check.stdout, end='')
if check.stderr:
    print(check.stderr, end='', file=sys.stderr)
check.check_returncode()

### 조작 방식 선택

`DEVICE`를 `sliders`, `quest`, `spacemouse` 중 하나로 설정하세요. Quest를 쓸 때는 `QUEST_HOST`에 헤드셋에서 접속할 PC의 LAN 주소를 입력하세요. SpaceMouse는 USB 수신기나 케이블로 PC에 연결합니다. 장치별 설정은 [UR5e 실행 안내](../README.md)를 참고하세요.

Quest와 SpaceMouse 모드에서는 **UR5e 조작 패널**이 함께 열립니다. 미리보기로 시작하려면 `START_IN_DRY_RUN = True`로 설정하세요. **DRY RUN**에서 클러치를 누른 채 캡이나 컨트롤러를 움직이면 가상 팔이 따라갑니다. 놓으면 가상 팔이 현재 시뮬레이션 로봇의 자세로 돌아가고, 다시 누를 때도 그 자세에서 시작합니다. **SIM RUN**에서는 시뮬레이션 로봇을 직접 조작합니다. Quest는 선택한 손의 grip을 사용합니다. SpaceMouse는 `BUTTON`으로 선택한 버튼(기본 1번)을 한 번 눌렀다 놓은 뒤 다시 누른 채 캡을 움직이세요. 패널에 선택한 버튼의 누름·놓음 상태가 표시됩니다.

SpaceMouse의 **0번과 1번을 함께 누르면 RG2가 닫힙니다**. 가상 그리퍼는 `GRIPPER_FORCE_N`과 현재 개도에 해당하는 제조사 속도 곡선을 따릅니다. 40 N에서는 110 mm부터 완전히 닫는 데 약 1.1–1.2초, 50 mm부터 약 0.4초가 걸립니다. 하나를 놓으면 **DRY RUN**은 보존된 SIM RUN 개도로 즉시 돌아갑니다. **SIM RUN**은 현재 개도에서 패널의 **열림 목표 폭**(기본 75 mm)까지 움직입니다. 값을 바꾸면 다음 열기부터 적용됩니다. 두 버튼을 누른 채 캡을 움직여 팔도 이동할 수 있습니다.

In [ ]:
import signal

DEVICE = 'sliders'
QUEST_HOST = ''
HAND = 'right'
BUTTON = 1
GRIPPER_FORCE_N = 40.0
START_IN_DRY_RUN = False

if DEVICE == 'sliders':
    command = simulator
else:
    command = [sys.executable, '-u', '-m', 'vai_robotics.ur5e.teleop',
               '--backend', 'sim', '--input', DEVICE]
    command += ['--gripper-force-n', str(GRIPPER_FORCE_N)]
    if START_IN_DRY_RUN:
        command.append('--dry-run')
    if DEVICE == 'quest':
        if not QUEST_HOST:
            raise ValueError('QUEST_HOST에 PC의 LAN 주소를 입력하세요.')
        command += ['--quest-host', QUEST_HOST, '--hand', HAND]
    elif DEVICE == 'spacemouse':
        command += ['--button', str(BUTTON)]

with subprocess.Popen(command, cwd=repo_root, stdout=subprocess.PIPE,
                      stderr=subprocess.STDOUT, text=True) as process:
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait():
            raise RuntimeError(f'시뮬레이터 종료 코드 {process.returncode}')
    except KeyboardInterrupt:
        if process.poll() is None:
            process.send_signal(signal.SIGINT)
            process.wait()